# Práctica — Clase 5: comparar inicializadores en self-attention

## Contexto

En el notebook de clase entrenaste un mini traductor EN→ES cuya capa
`SelfAttention` inicializa sus matrices `Wq`, `Wk`, `Wv` con el inicializador
`glorot_uniform` (Xavier). Este es el **ejercicio real** propuesto en el
material fuente (notebook "Transformers with Keras" de IBM, ejercicio 1):
entrenar la misma arquitectura pero cambiando el inicializador a
`he_uniform`, y comparar las curvas de pérdida de ambos entrenamientos en un
mismo gráfico.

Los inicializadores deciden con qué valores aleatorios arrancan los pesos
antes de entrenar. Glorot (`glorot_uniform`) está pensado para activaciones
simétricas como `tanh`; He (`he_uniform`) está pensado para `relu` y
variantes. Aquí no usamos `relu` en la capa de atención, así que este
ejercicio también te sirve para comprobar empíricamente si el cambio importa
o no en este caso concreto.

## Paso 0: instalar dependencias y preparar los datos (igual que en la clase)

In [ ]:
!pip install tensorflow_cpu==2.17.1
!pip install matplotlib==3.9.2

import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Concatenate, Layer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import backend as K
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter('ignore', FutureWarning)

In [ ]:
# Mismos datos que en el notebook de clase
input_texts = [
    "Hello.", "How are you?", "I am learning machine translation.",
    "What is your name?", "I love programming."
]
target_texts = [
    "Hola.", "¿Cómo estás?", "Estoy aprendiendo traducción automática.",
    "¿Cuál es tu nombre?", "Me encanta programar."
]
target_texts = ["startseq " + x + " endseq" for x in target_texts]

input_tokenizer = Tokenizer()
input_tokenizer.fit_on_texts(input_texts)
input_sequences = input_tokenizer.texts_to_sequences(input_texts)

output_tokenizer = Tokenizer()
output_tokenizer.fit_on_texts(target_texts)
output_sequences = output_tokenizer.texts_to_sequences(target_texts)

input_vocab_size = len(input_tokenizer.word_index) + 1
output_vocab_size = len(output_tokenizer.word_index) + 1

max_input_length = max([len(seq) for seq in input_sequences])
max_output_length = max([len(seq) for seq in output_sequences])

input_sequences = pad_sequences(input_sequences, maxlen=max_input_length, padding='post')
output_sequences = pad_sequences(output_sequences, maxlen=max_output_length, padding='post')

decoder_input_data = output_sequences[:, :-1]
decoder_output_data = output_sequences[:, 1:]
decoder_output_data = np.array([np.eye(output_vocab_size)[seq] for seq in decoder_output_data])

## Andamiaje: capa de atención y arquitectura parametrizables

Para no repetir la arquitectura completa dos veces, esta versión de
`SelfAttention` recibe el nombre del inicializador como parámetro, y
`build_model(initializer)` arma el modelo completo (encoder + decoder +
atención) usando ese inicializador. Es exactamente el mismo modelo que en la
clase — solo que ahora puedes reutilizarlo con distintos inicializadores sin
copiar y pegar código.

In [ ]:
class SelfAttention(Layer):
    def __init__(self, initializer='glorot_uniform', **kwargs):
        super(SelfAttention, self).__init__(**kwargs)
        self.initializer = initializer

    def build(self, input_shape):
        feature_dim = input_shape[0][-1]

        self.Wq = self.add_weight(
            shape=(feature_dim, feature_dim),
            initializer=self.initializer,
            trainable=True,
            name='Wq'
        )
        self.Wk = self.add_weight(
            shape=(feature_dim, feature_dim),
            initializer=self.initializer,
            trainable=True,
            name='Wk'
        )
        self.Wv = self.add_weight(
            shape=(feature_dim, feature_dim),
            initializer=self.initializer,
            trainable=True,
            name='Wv'
        )
        super(SelfAttention, self).build(input_shape)

    def call(self, inputs):
        q, k, v = inputs
        q = K.dot(q, self.Wq)
        k = K.dot(k, self.Wk)
        v = K.dot(v, self.Wv)

        scores = K.batch_dot(q, k, axes=[2, 2])
        dk = K.cast(K.shape(k)[-1], dtype=K.floatx())
        scores = scores / K.sqrt(dk)

        attention_weights = K.softmax(scores, axis=-1)
        output = K.batch_dot(attention_weights, v)
        return output


def build_model(initializer='glorot_uniform'):
    """Construye y compila el modelo encoder-decoder + self-attention,
    usando `initializer` para las matrices Wq, Wk, Wv de la capa de atención."""
    encoder_inputs = Input(shape=(max_input_length,))
    encoder_embedding = Embedding(input_vocab_size, 256)(encoder_inputs)
    encoder_lstm = LSTM(256, return_sequences=True, return_state=True)
    encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)
    encoder_states = [state_h, state_c]

    decoder_inputs = Input(shape=(max_output_length - 1,))
    decoder_embedding = Embedding(output_vocab_size, 256)(decoder_inputs)
    decoder_lstm = LSTM(256, return_sequences=True, return_state=True)
    decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

    self_attention = SelfAttention(initializer=initializer)
    attention_output = self_attention([decoder_outputs, encoder_outputs, encoder_outputs])

    decoder_concat = Concatenate(axis=-1)([decoder_outputs, attention_output])
    decoder_dense = Dense(output_vocab_size, activation='softmax')
    decoder_outputs = decoder_dense(decoder_concat)

    model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

## Modelo base: `glorot_uniform` (ya entrenado para ti)

Esta celda entrena el modelo de referencia, igual que en el notebook de
clase, y guarda su historial en `history_glorot`.

In [ ]:
model_glorot = build_model(initializer='glorot_uniform')

history_glorot = model_glorot.fit(
    [input_sequences, decoder_input_data],
    decoder_output_data,
    epochs=100,
    batch_size=16,
    verbose=0
)

print("Pérdida final (glorot_uniform):", history_glorot.history['loss'][-1])

## Ejercicio: entrena con `he_uniform` y compara

**Enunciado**: crea un segundo modelo idéntico al de arriba, pero usando
`initializer='he_uniform'` en la capa de self-attention. Entrénalo con los
mismos hiperparámetros (100 épocas, batch_size=16), guarda su historial en
`history_he`, y grafica ambas curvas de pérdida (`history_glorot` vs
`history_he`) en el mismo gráfico para comparar cuál converge más rápido o
más bajo.

Pistas:
- Ya tienes la función `build_model(initializer=...)` lista para reutilizar.
- `history.history['loss']` es la lista de valores de pérdida por época.
- Usa `plt.plot(..., label=...)` y `plt.legend()` para distinguir las curvas.

In [ ]:
# TODO: construye un segundo modelo usando initializer='he_uniform'
model_he = None  # TODO: usa build_model('he_uniform')

# TODO: entrena el modelo con los mismos hiperparámetros que el modelo base
# (epochs=100, batch_size=16) y guarda el resultado en history_he
history_he = None

# TODO: grafica history_glorot.history['loss'] y history_he.history['loss']
# en el mismo gráfico, con leyenda, título y etiquetas de los ejes

<details>
<summary>💡 Ver solución</summary>

```python
# Construimos el segundo modelo con he_uniform
model_he = build_model(initializer='he_uniform')

# Lo entrenamos con los mismos hiperparámetros que el modelo base
history_he = model_he.fit(
    [input_sequences, decoder_input_data],
    decoder_output_data,
    epochs=100,
    batch_size=16,
    verbose=0
)

print("Pérdida final (he_uniform):", history_he.history['loss'][-1])

# Comparamos ambas curvas de pérdida en el mismo gráfico
plt.plot(history_glorot.history['loss'], label='glorot_uniform', color='red')
plt.plot(history_he.history['loss'], label='he_uniform', color='blue')
plt.title('Pérdida de entrenamiento: glorot_uniform vs he_uniform')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.legend()
plt.show()
```

Con un dataset de solo 5 frases, es normal que ambas curvas converjan a una
pérdida muy baja y que la diferencia entre inicializadores sea pequeña o
inconsistente entre ejecuciones (los pesos arrancan aleatorios). El valor del
ejercicio está en el proceso — parametrizar el modelo, entrenar dos
variantes y comparar curvas — más que en un ganador claro.
</details>

## Para seguir

- Vuelve al notebook de clase:
  [clase05_transformers_atencion.ipynb](../notebooks_clase/clase05_transformers_atencion.ipynb).
- Repasa la teoría en
  [manual/08_transformers_atencion.md](../manual/08_transformers_atencion.md).
- Si quieres ir más allá, el ejercicio 2 del material original propone
  comparar el optimizador `adam` contra `adagrad` con el mismo patrón que
  usaste aquí para los inicializadores.